In [3]:
import sys; sys.path.append("..")

import os
import numpy as np
import pandas as pd
from openbabel import openbabel as ob
from sqlalchemy.orm import Session
from src.featurizers import FunctionFeaturizer
from src import utils

from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
engine = create_engine("sqlite:///{}".format("../main.db"))
session = sessionmaker(bind=engine)()

In [11]:
def target_featurizer(session: Session, sid: str):
    props_df = pd.read_sql(f"SELECT property,source,value FROM structure_properties WHERE structure='{sid}' AND source LIKE 'parser%'", session.connection())
    props_df = props_df.pivot_table(index=["property", "source"], values="value", aggfunc="first").unstack("source")
    props_df.columns = ["{}".format(col[1].split("/")[-1]) for col in props_df.columns]
    if any([col not in props_df.columns for col in ["S1", "S3", "S5"]]):
        return [None, None, None, None]
    props_df = props_df[["S1", "S3", "S5"]]
    spin_state = props_df.loc["final_energy", :].idxmin()
    energies = props_df.loc["final_energy", :]
    energies = (energies - energies.min()) * 27.2114 # convert to eV from Ha
    return [spin_state] + energies.tolist()



feat = FunctionFeaturizer(["state", "S1", "S3", "S5"], target_featurizer, navalue=None)
all_data = feat.featurize(session, utils.sids_by_type(session))
all_data

,state,S1,S3,S5
ADIQAI,S5,0.952712,0.545094,0.0
AKOTUQ,S1,0.0,0.56074,2.226144
ALITEU,None,None,None,None
ANUYOY,None,None,None,None
APANUC,S1,0.0,1.590224,3.452527
...,...,...,...,...
ZONQOI,None,None,None,None
ZONXEF,None,None,None,None
ZOXQUA,S1,0.0,1.806496,3.741497
ZUTQEK,None,None,None,None


In [32]:
valid_sids = []
for fname in os.listdir("../data/curated_xyz/porphyrins/"):
    s = fname.split("_")[0]
    valid_sids.append(s)
valid_sids

['ADIQAI',
 'AKOTUQ',
 'ALITEU',
 'ANUYOY',
 'APANUC',
 'AQOYAI',
 'ARODOC',
 'ASIPIB',
 'ASIPOH',
 'ASIPUN',
 'ATEWUT',
 'ATUSOX',
 'AWIQEC',
 'AWOMOP',
 'AXUWUL',
 'AZIVEK',
 'BADREE',
 'BAYDEL',
 'BEDCAO',
 'BEGLUU',
 'BITPFE',
 'BIYXIS',
 'BOHNUK',
 'BORJEY',
 'BORJIC',
 'BORJOI',
 'BORKEZ',
 'BUFMUN',
 'BUGJIX',
 'BUSFON',
 'BUVKOV',
 'BUXVOG',
 'CAHCOH',
 'CAQMUF',
 'CAQNEO',
 'CASNOC',
 'CATYEE',
 'CEBMAZ',
 'CEHVIX',
 'CEHVUJ',
 'CETWON',
 'CEZKEX01',
 'CIBTIS',
 'CICQAG',
 'CONQOL',
 'DAFBOC',
 'DAFCIX',
 'DAFCOD',
 'DAGSUC',
 'DAHSUB',
 'DEFKIM',
 'DEGQUD',
 'DITDES',
 'DIVVIO',
 'DIXXIS',
 'DOFXOL',
 'DOJPAV01',
 'DOKROM',
 'DOWCAT',
 'DUQFAY',
 'DUVSOE',
 'DUXROF',
 'EACWAG',
 'EBOMIS',
 'EBOQUH',
 'EBORES01',
 'EBORES',
 'ECUBAH',
 'EDOHAJ',
 'EDOHIR',
 'EKETUJ',
 'EQUFON',
 'EREDIO',
 'EREDUA',
 'EREFAI',
 'ETISEI',
 'ETISIM',
 'ETUKOW',
 'EWURUL',
 'EWUSAS',
 'FAMTAS',
 'FAMTEW',
 'FAMTIA',
 'FAPCAC',
 'FARJIU',
 'FATRAW',
 'FAVGUE',
 'FEMQIX',
 'FEMQOD',
 'FEMQUJ',
 'FE

In [34]:
df = all_data.dropna()
df = df[df.index.isin(valid_sids)]
sorted_energeies = np.sort(df.iloc[:, 1:].values, axis=1)
df["first_shift"] = sorted_energeies[:, 1] - sorted_energeies[:, 0]
df["second_shift"] = sorted_energeies[:, 2] - sorted_energeies[:, 1]
df = df.sort_values(by="first_shift")

In [39]:
df

,state,S1,S3,S5,first_shift,second_shift
TUBJAB,S3,0.010312,0.0,0.638755,0.010312,0.628443
SOMDEF,S3,0.710209,0.0,0.010714,0.010714,0.699494
IMELIV,S3,0.018636,0.0,0.940075,0.018636,0.921439
GOBBUV01,S3,0.428228,0.0,0.096423,0.096423,0.331804
GOBBUV,S3,0.427992,0.0,0.097161,0.097161,0.330831
...,...,...,...,...,...,...
TEMFAS01,S1,0.0,2.056946,3.949229,2.056946,1.892283
AZIVEK,S1,0.0,2.112552,4.110151,2.112552,1.997599
USIVAV,S1,0.0,2.134933,4.137132,2.134933,2.002199
GIXCIB,S1,0.0,2.337107,4.24187,2.337107,1.904763


In [40]:
from shutil import copyfile

def send_to_benchmark(sid: str):
    xyz = os.path.join("../data/curated_xyz/porphyrins", sid + "_0.xyz")
    target = os.path.join("../data/benchmark/xyz", sid + "_0.xyz")
    if not os.path.isfile(target):
        copyfile(xyz, target)

bad_sids = [
    "GIXCIB", # 2 metlas
    "TUBJAB",  # unsaturated OH meso groups
    "GOBBUV01", # duplicate with GOBBUV
]

for sid in df.index[:3 + len(bad_sids)].tolist() + df.index[-3:].tolist():
    if not sid in bad_sids:
        send_to_benchmark(sid)


In [ ]:
ajr = df.copy